# Практика: Структурированный вывод с GigaChat API

В этом ноутбуке мы научимся получать от LLM не просто текст, а **структурированные данные** в формате JSON, которые можно сразу использовать в коде.

**Финальное задание:** построить мини-пайплайн для автоматического разбора резюме — программа будет извлекать из произвольного текста структурированный профиль кандидата.

## 0. Получение ключа авторизации GigaChat

**Пошаговая инструкция:**

1. Зарегистрируйтесь на [developers.sber.ru](https://developers.sber.ru)
2. Создайте проект **GigaChat API** (для физлиц, тариф Free)
3. В настройках проекта получите **Client ID** и **Client Secret**,
   из которых собирается ключ авторизации (Base64-строка)
4. Скопируйте ключ авторизации

> **Важно:** Python-клиенты не используют системное хранилище сертификатов,
> поэтому сертификат НУЦ Минцифры подключается в SDK напрямую через
> `ca_bundle_file`. Бесплатный тариф имеет лимиты по частоте запросов.

Создайте файл `.env` в той же папке, что и этот ноутбук:


```
GIGACHAT_CREDENTIALS=ВАШ_КЛЮЧ_АВТОРИЗАЦИИ
GIGACHAT_CA_BUNDLE_FILE=certs/Russian_Trusted_Root_CA.cer
```

## 1. Установка и настройка клиента

In [12]:
!pip install gigachat python-dotenv

import os
import json
import time

from dotenv import load_dotenv
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

load_dotenv()

api_key = os.getenv('GIGACHAT_CREDENTIALS')
ca_bundle = os.getenv('GIGACHAT_CA_BUNDLE_FILE')

client = GigaChat(
    credentials=api_key,
    scope='GIGACHAT_API_PERS',
    ca_bundle_file=ca_bundle,
    timeout=60.0,
)

print(f'Ключ GigaChat загружен: {api_key[:10] if api_key else "Не найден"}...')
print(f'CA-бандл: {ca_bundle}')

Ключ GigaChat загружен: MDFhMTYzMT...
CA-бандл: certs/Russian_Trusted_Root_CA.cer


## 2. Проблема: LLM возвращает неструктурированный текст

Попросим модель извлечь данные из текста обычным запросом — и посмотрим, почему такой вывод неудобен для программы.

In [13]:
resume_text = """
Привет! Меня зовут Анна Соколова, мне 28 лет. Последние 4 года работаю
бэкенд-разработчиком в финтех-компании. Основной стек: Python, FastAPI,
PostgreSQL, Redis. Есть опыт с Docker и Kubernetes. Закончила МГТУ им. Баумана
по специальности «Прикладная математика». Ищу позицию Senior Python Developer
с зарплатой от 300 000 рублей. Готова к удалённой работе или гибриду.
"""

# Обычный запрос без требования структуры — контракт v1
chat = Chat(
    model='GigaChat-2',
    messages=[
        Messages(
            role=MessagesRole.USER,
            content=f'Извлеки основную информацию из резюме:\n{resume_text}',
        )
    ],
)

response = client.chat(chat)

raw_output = response.choices[0].message.content
print('Вывод модели:')
print(raw_output)
print()

# Попытка использовать вывод в коде — неудача
print('Попытка получить зарплату из ответа:')
try:
    data = json.loads(raw_output)
    print(data['salary'])
except Exception as e:
    print(f'Ошибка: {e}')
    print('Неструктурированный текст нельзя напрямую использовать в коде!')

Вывод модели:
**Основная информация из резюме Анны Соколовой:**

- **ФИО:** Анна Соколова  
- **Возраст:** 28 лет  
- **Опыт работы:** 4 года бэкенд-разработчиком  
- **Место работы:** Финтех-компания  
- **Ключевые технологии:**  
  - Python  
  - FastAPI  
  - PostgreSQL  
  - Redis  
  - Docker  
  - Kubernetes  
- **Образование:**  
  - МГТУ им. Баумана, специальность «Прикладная математика»  
- **Цель:** позиция Senior Python Developer  
- **Желаемый уровень зарплаты:** от 300 000 руб.  
- **Готова к формату работы:** удалённая работа или гибридный график

Попытка получить зарплату из ответа:
Ошибка: Expecting value: line 1 column 1 (char 0)
Неструктурированный текст нельзя напрямую использовать в коде!


## 3. Решение: структурированный вывод (json_schema)

У GigaChat нет режима `json_object` как у Groq — вместо «попроси JSON» здесь передаётся **полная JSON-схема** в `response_format.schema` и параметр
`strict: True`. Схема гарантирует наличие всех полей в ответе, а отсутствующие в тексте данные модель вернёт как `null` (для этого в схеме у полей тип `["number", "null"]`).

In [14]:
system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
Схема ответа:
{
  "name": string,
  "age": number или null,
  "experience_years": number,
  "current_position": string,
  "skills": [string],
  "education": string или null,
  "target_position": string,
  "salary_from": number или null,
  "work_format": string
}
"""

# JSON-схема профиля кандидата — контрактуем ответ модели
resume_schema = {
    'type': 'object',
    'properties': {
        'name': {'type': 'string'},
        'age': {'type': ['number', 'null']},
        'experience_years': {'type': 'number'},
        'current_position': {'type': 'string'},
        'skills': {'type': 'array', 'items': {'type': 'string'}},
        'education': {'type': ['string', 'null']},
        'target_position': {'type': ['string', 'null']},
        'salary_from': {'type': ['number', 'null']},
        'work_format': {'type': 'string'},
    },
    'required': [
        'name', 'age', 'experience_years', 'current_position', 'skills',
        'education', 'target_position', 'salary_from', 'work_format',
    ],
    'additionalProperties': False,
}

chat = Chat(
    model='GigaChat-2',
    messages=[
        Messages(role=MessagesRole.SYSTEM, content=system_prompt),
        Messages(role=MessagesRole.USER, content=f'Обработай резюме:\n{resume_text}'),
    ],
    response_format={
        'type': 'json_schema',
        'schema': resume_schema,
        'strict': True,
    },
)

response = client.chat(chat)

candidate = json.loads(response.choices[0].message.content)

print('Структурированный профиль кандидата:')
print(json.dumps(candidate, ensure_ascii=False, indent=2))
print()

# Теперь данные легко использовать в коде
print(f"Имя: {candidate['name']}")
print(f"Опыт: {candidate['experience_years']} лет")
print(f"Навыки: {', '.join(candidate['skills'])}")
print(f"Ожидаемая зарплата: {candidate['salary_from']:,} руб.".replace(',', ' '))

Структурированный профиль кандидата:
{
  "name": "Анна Соколова",
  "age": 28,
  "experience_years": 4,
  "current_position": "бэкенд-разработчик",
  "skills": [
    "Python",
    "FastAPI",
    "PostgreSQL",
    "Redis",
    "Docker",
    "Kubernetes"
  ],
  "education": "МГТУ им. Баумана, специальность «Прикладная математика»",
  "target_position": "Senior Python Developer",
  "salary_from": 300000,
  "work_format": "удалённая работа или гибрид"
}

Имя: Анна Соколова
Опыт: 4 лет
Навыки: Python, FastAPI, PostgreSQL, Redis, Docker, Kubernetes
Ожидаемая зарплата: 300 000 руб.


## 4. Пайплайн: обработка нескольких резюме

Завернём логику в функцию и запустим её на нескольких резюме сразу.

In [15]:
from gigachat.exceptions import RateLimitError

def parse_resume(text: str, attempts: int = 3) -> dict | None:
    """Извлекает структурированные данные из текста резюме."""
    system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON. Если целевая позиция в резюме не указана — верни null.
"""

    chat = Chat(
        model='GigaChat-2',
        messages=[
            Messages(role=MessagesRole.SYSTEM, content=system_prompt),
            Messages(role=MessagesRole.USER, content=f'Обработай резюме:\n{text}'),
        ],
        response_format={
            'type': 'json_schema',
            'schema': resume_schema,  
            'strict': True,
        },
    )

    for attempt in range(1, attempts + 1):
        try:
            response = client.chat(chat)
            return json.loads(response.choices[0].message.content)
        except RateLimitError as e:
            wait = getattr(e, 'retry_after', None) or 5
            print(f'Лимит запросов (попытка {attempt}/{attempts}). Ждём {wait} сек...')
            time.sleep(wait)
        except Exception as e:
            print(f'Ошибка: {e}')
            return None
    return None


resumes = [
    """
    Иван Петров, 32 года. 7 лет в разработке игр на Unity и C#.
    Делал мобильные игры для iOS/Android, есть опыт с Blender.
    Ищу Lead Game Developer, готов к офису в Москве. Зарплата от 250 000.
    """,
    """
    Мария Иванова. Data Scientist с опытом 3 года.
    Стек: Python, pandas, scikit-learn, PyTorch, SQL.
    Работала в e-commerce над рекомендательными системами.
    Хочу ML Engineer, только удалёнка, зарплата не принципиальна.
    """,
    """
    Дмитрий Козлов, 25 лет, junior frontend. Знаю React, TypeScript, CSS.
    Полгода коммерческого опыта в веб-студии. Ищу работу в продуктовой компании.
    Рассматриваю от 120 000, гибрид или удалёнка.
    """,
    """
    Екатерина Смирнова, 29 лет. DevOps-инженер, 5 лет опыта.
    Умею: Kubernetes, Docker, Terraform, AWS, CI/CD (GitLab, GitHub Actions).
    Работала в банковском секторе, есть опыт с highload-системами.
    Ищу Senior DevOps, удалёнка обязательно. Зарплата от 300 000.
    """,
    """
    Алексей Новиков, 35 лет. Fullstack-разработчик, 10 лет опыта.
    Стек: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker.
    Последние 3 года — тимлид команды из 6 человек в SaaS-компании.
    Рассматриваю роль Engineering Manager или Staff Engineer. От 400 000, гибрид.
    """,
    """
    Юлия Белова. QA Engineer, 4 года в тестировании.
    Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests).
    Опыт в agile-командах, знакома с нагрузочным тестированием (Locust).
    Ищу Middle/Senior QA, гибрид, зарплата от 180 000.
    """,
    """
    Сергей Орлов, 27 лет. Android-разработчик, 3 года опыта.
    Kotlin, Jetpack Compose, Room, Retrofit, Coroutines. Публиковал 2 приложения в Google Play.
    Немного знаю iOS (Swift). Ищу Middle Android Developer в продуктовую компанию.
    Только удалёнка. Зарплата от 210 000.
    """,
]

results = []
for i, resume in enumerate(resumes, 1):
    print(f'Обрабатываю резюме {i}/{len(resumes)}...')
    parsed = parse_resume(resume)
    if parsed:
        results.append(parsed)
    time.sleep(1)  # бережём лимиты бесплатного тарифа

print(f'\nОбработано: {len(results)} из {len(resumes)} резюме')
print()
for r in results:
    salary = f"{r['salary_from']:,}".replace(',', ' ') + ' руб.' if r.get('salary_from') else 'не указана'
    print(f"• {r['name']} | {r['target_position']} | опыт {r['experience_years']} л. | зарплата: {salary}")

Обрабатываю резюме 1/7...
Обрабатываю резюме 2/7...
Обрабатываю резюме 3/7...
Обрабатываю резюме 4/7...
Обрабатываю резюме 5/7...
Обрабатываю резюме 6/7...
Обрабатываю резюме 7/7...

Обработано: 7 из 7 резюме

• Иван Петров | Lead Game Developer | опыт 7 л. | зарплата: 250 000 руб.
• Мария Иванова | ML Engineer | опыт 3 л. | зарплата: не указана
• Дмитрий Козлов | junior frontend | опыт 0.5 л. | зарплата: 120 000 руб.
• Екатерина Смирнова | Senior DevOps | опыт 5 л. | зарплата: 300 000 руб.
• Алексей Новиков | None | опыт 10 л. | зарплата: 400 000 руб.
• Юлия Белова | Middle/Senior QA | опыт 4 л. | зарплата: 180 000 руб.
• Сергей Орлов | Middle Android Developer | опыт 3 л. | зарплата: 210 000 руб.


## 5. Фильтрация и анализ собранных данных

Структурированные данные позволяют легко фильтровать кандидатов без дополнительных запросов к модели.

Ниже показан один пример фильтрации — изучите его, затем **выполните самостоятельные задания** в следующей ячейке.


In [16]:
# Пример: кандидаты с опытом 3+ лет и зарплатой до 300 000
filtered = [
    r for r in results
    if r['experience_years'] >= 3
    and (r['salary_from'] is None or r['salary_from'] <= 300_000)
]

print(f'Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): {len(filtered)}')
for r in filtered:
    print(f"  → {r['name']}: {', '.join(r['skills'][:3])}...")

print()

# Самый опытный кандидат
if results:
    top = max(results, key=lambda r: r['experience_years'])
    print(f"Самый опытный: {top['name']} ({top['experience_years']} лет опыта)")

# Все упомянутые навыки
all_skills = [skill for r in results for skill in r['skills']]
unique_skills = sorted(set(all_skills))
print(f'\nВсе навыки в пуле: {len(unique_skills)}')
print(', '.join(unique_skills))


Кандидаты (опыт ≥ 3 лет, зарплата ≤ 300 000): 5
  → Иван Петров: Разработка игр на Unity, Программирование на C#, Создание мобильных игр для iOS/Android...
  → Мария Иванова: Python, pandas, scikit-learn...
  → Екатерина Смирнова: Kubernetes, Docker, Terraform...
  → Юлия Белова: Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests), Опыт в agile-командах...
  → Сергей Орлов: Kotlin, Jetpack Compose, Room...

Самый опытный: Алексей Новиков (10 лет опыта)

Все навыки в пуле: 30
API-тестирование (Postman, requests), AWS, CI/CD (GitLab, GitHub Actions), CSS, Coroutines, Docker, Jetpack Compose, Kotlin, Kubernetes, PostgreSQL, PyTorch, Python, Python (Django, FastAPI), React, Redis, Retrofit, Room, SQL, Terraform, TypeScript, Vue.js, pandas, scikit-learn, Автотесты на Python (pytest, Selenium, Playwright), Знакома с нагрузочным тестированием (Locust), Опыт в agile-командах, Опыт работы с Blender, Программирование на C#, Разработка игр на Unity, Создание м

### Задание: фильтрация кандидатов

Используя список `results`, выполните следующие задачи в ячейке ниже:

1. **Удалёнщики-синьоры** — найдите кандидатов, у которых `work_format` содержит слово `удалёнка` и `experience_years >= 4`.
2. **Бюджетный найм** — выведите всех кандидатов с зарплатными ожиданиями не выше 230 000 руб. (или без указанной зарплаты).
3. **Python-разработчики** — отберите кандидатов, у которых в `skills` есть `Python`.
4. **Топ-3 по опыту** — выведите трёх самых опытных кандидатов в порядке убывания стажа.
5. **Статистика зарплат** — посчитайте среднюю, минимальную и максимальную ожидаемую зарплату среди тех, кто её указал.


In [20]:
# 0. Санитарная проверка: типы полей, по которым будем фильтровать
for r in results:
    problems = []
    if not isinstance(r.get('experience_years'), (int, float)):
        problems.append('experience_years')
    if not isinstance(r.get('skills'), list):
        problems.append('skills')
    if r.get('salary_from') is not None and not isinstance(r['salary_from'], (int, float)):
        problems.append('salary_from')
    if not isinstance(r.get('work_format'), str):
        problems.append('work_format')
    print(f"{r['name']:20} {'OK' if not problems else 'ПРОБЛЕМЫ: ' + ', '.join(problems)}")

# Задание 1: удалёнщики с опытом 4+ лет
remote_seniors = [
    r for r in results
    if 'удалёнка' in r['work_format'].lower() and r['experience_years'] >= 4
]
print(f'\n1. Удалёнщики с опытом 4+ лет: {len(remote_seniors)}')
for r in remote_seniors:
    print(f"   → {r['name']} ({r['experience_years']} лет, {r['work_format']})")

# Задание 2: зарплатные ожидания ≤ 230 000 руб. (или не указаны)
budget_hires = [
    r for r in results
    if r['salary_from'] is None or r['salary_from'] <= 230_000
]
print(f'\n2. Бюджетный найм (≤ 230 000 руб. или не указана): {len(budget_hires)}')
for r in budget_hires:
    s = f"{r['salary_from']:,} руб.".replace(',', ' ') if r['salary_from'] else 'не указана'
    print(f"   → {r['name']} ({s})")

# Задание 3: кандидаты с Python в навыках
python_devs = [
    r for r in results
    if any('python' in s.lower() for s in r['skills'])
]
print(f'\n3. Кандидаты с Python: {len(python_devs)}')
for r in python_devs:
    print(f"   → {r['name']}: {', '.join(r['skills'])}")

# Задание 4: топ-3 по опыту
top3 = sorted(results, key=lambda r: r['experience_years'], reverse=True)[:3]
print('\n4. Топ-3 по опыту:')
for i, r in enumerate(top3, 1):
    print(f"   {i}. {r['name']} — {r['experience_years']} лет")

# Задание 5: статистика зарплат среди указавших
salaries = [r['salary_from'] for r in results if r['salary_from'] is not None]
print(f'\n5. Статистика зарплат (указали {len(salaries)} из {len(results)}):')
print(f"   средняя:      {sum(salaries) / len(salaries):,.0f} руб.".replace(',', ' '))
print(f"   минимальная:  {min(salaries):,} руб.".replace(',', ' '))
print(f"   максимальная: {max(salaries):,} руб.".replace(',', ' '))

Иван Петров          OK
Мария Иванова        OK
Дмитрий Козлов       OK
Екатерина Смирнова   OK
Алексей Новиков      OK
Юлия Белова          OK
Сергей Орлов         OK

1. Удалёнщики с опытом 4+ лет: 1
   → Екатерина Смирнова (5 лет, удалёнка)

2. Бюджетный найм (≤ 230 000 руб. или не указана): 4
   → Мария Иванова (не указана)
   → Дмитрий Козлов (120 000 руб.)
   → Юлия Белова (180 000 руб.)
   → Сергей Орлов (210 000 руб.)

3. Кандидаты с Python: 3
   → Мария Иванова: Python, pandas, scikit-learn, PyTorch, SQL
   → Алексей Новиков: Python (Django, FastAPI), Vue.js, PostgreSQL, Redis, Docker
   → Юлия Белова: Автотесты на Python (pytest, Selenium, Playwright), API-тестирование (Postman, requests), Опыт в agile-командах, Знакома с нагрузочным тестированием (Locust)

4. Топ-3 по опыту:
   1. Алексей Новиков — 10 лет
   2. Иван Петров — 7 лет
   3. Екатерина Смирнова — 5 лет

5. Статистика зарплат (указали 6 из 7):
   средняя:      243 333 руб.
   минимальная:  120 000 руб.
   максималь

## 6. Сравнение моделей

GigaChat API предоставляет доступ к нескольким моделям с разными характеристиками. Сравним их по **скорости** и **качеству** извлечения структурированных данных.

В отличие от Groq, где модели различаются в первую очередь числом параметров (8B vs 70B), линейка GigaChat различается **поколением** и **классом** модели. Список получен методом `get_models()` для текущего ключа:


| Модель | Поколение / класс | Ожидаемая роль |
|---|---|---|
| `GigaChat-2` | 2, базовая | Быстрая модель для повседневных задач |
| `GigaChat-3-Lightning` | 3, скоростная | Минимальная задержка ответа |
| `GigaChat-2-Pro` | 2, Pro | Ресурсоёмкие задачи |
| `GigaChat-3-Ultra` | 3, топовая | Флагман для самых сложных задач |

Актуальный список моделей доступен методом `get_models()` или в документации: https://developers.sber.ru/docs/ru/gigachat/models


In [22]:

import time

# Резюме для теста
test_resume = """
Андрей Захаров, 31 год. Backend-разработчик, 6 лет опыта.
Основной стек: Go, Python, gRPC, Kafka, PostgreSQL, Kubernetes.
Строил высоконагруженные сервисы в финтех и логистике.
Ищу Senior/Lead Backend Engineer. Удалёнка, зарплата от 380 000.
"""

system_prompt = """
Ты — HR-ассистент. Извлекай данные из резюме и возвращай строго в формате JSON.
"""

# Линейка GigaChat: две оси сравнения — поколение (2 vs 3) и класс (базовая vs Pro/Ultra)
models_to_compare = [
    'GigaChat-2',
    'GigaChat-3-Lightning',
    'GigaChat-2-Pro',
    'GigaChat-3-Ultra',
]

comparison_results = []

for model_name in models_to_compare:
    try:
        start = time.time()
        chat = Chat(
            model=model_name,
            messages=[
                Messages(role=MessagesRole.SYSTEM, content=system_prompt),
                Messages(role=MessagesRole.USER, content=f'Обработай резюме:\n{test_resume}'),
            ],
            response_format={
                'type': 'json_schema',
                'schema': resume_schema,
                'strict': True,
            },
        )
        response = client.chat(chat)
        elapsed = time.time() - start

        parsed = json.loads(response.choices[0].message.content)
        usage = response.usage

        comparison_results.append({
            'model': model_name,
            'time_sec': round(elapsed, 2),
            'total_tokens': usage.total_tokens,
            'tokens_per_sec': round(usage.completion_tokens / elapsed),
            'skills_found': len(parsed.get('skills', [])),
            'parsed': parsed,
        })
        print(f'OK {model_name}: {elapsed:.2f} с, {usage.total_tokens} токенов')
        time.sleep(1)  # пауза между запросами — бережём лимиты тарифа
    except Exception as e:
        print(f'ОШИБКА {model_name}: {e}')

print()
header = f"{'Модель':<20} {'Время':>8} {'Ток/с':>8} {'Навыков':>9}"
print(header)
print('-' * 50)
for r in comparison_results:
    row = f"{r['model']:<20} {r['time_sec']:>7.2f}с {r['tokens_per_sec']:>8} {r['skills_found']:>9}"
    print(row)


OK GigaChat-2: 1.33 с, 287 токенов
OK GigaChat-3-Lightning: 1.04 с, 200 токенов
OK GigaChat-2-Pro: 3.15 с, 272 токенов
OK GigaChat-3-Ultra: 1.26 с, 260 токенов

Модель                  Время    Ток/с   Навыков
--------------------------------------------------
GigaChat-2              1.33с      122         6
GigaChat-3-Lightning    1.04с      152         6
GigaChat-2-Pro          3.15с       46         6
GigaChat-3-Ultra        1.26с      112         6


In [23]:
# Сравнение качества извлечения между моделями
fields = ['name', 'experience_years', 'current_position', 'target_position', 'salary_from', 'work_format']

print(f"{'Поле':<20}", end='')
for r in comparison_results:
    print(f"{r['model']:>24}", end='')
print()
print('-' * 116)

for f in fields:
    print(f"{f:<20}", end='')
    for r in comparison_results:
        val = str(r['parsed'].get(f))
        print(f"{val[:22]:>24}", end='')
    print()

Поле                              GigaChat-2    GigaChat-3-Lightning          GigaChat-2-Pro        GigaChat-3-Ultra
--------------------------------------------------------------------------------------------------------------------
name                          Андрей Захаров          Андрей Захаров          Андрей Захаров          Андрей Захаров
experience_years                           6                       6                       6                       6
current_position         Backend-разработчик     Backend-разработчик     Backend-разработчик     Backend-разработчик
target_position       Senior/Lead Backend En  Senior/Lead Backend En  Senior/Lead Backend En  Senior/Lead Backend En
salary_from                           380000                  380000                  380000                  380000
work_format                         Удалёнка                Удалёнка                удалёнка                Удалёнка


### Задание: анализ результатов сравнения

После выполнения ячейки выше ответьте на вопросы (в новой ячейке типа Markdown или в комментариях):

1. Какая модель оказалась **самой быстрой** по времени ответа? А по числу токенов в секунду?
2. Все ли модели извлекли одинаковое количество навыков? Есть ли расхождения?
3. Когда стоит предпочесть большую модель (70B) маленькой (8B)? Приведите пример задачи.
4. **Бонус:** добавьте в список ещё одну модель из документации Groq и запустите сравнение заново.


---
## Самостоятельное задание: парсер вакансий

Вы видели, как разбирать **резюме**. Теперь сделайте то же самое для **вакансий**.

### Задача
Реализуйте функцию `parse_vacancy(text: str) -> dict | None`, которая принимает текст вакансии и возвращает структурированный JSON со следующими полями:

```json
{
  "position": "название должности",
  "company": "название компании или null",
  "required_skills": ["список обязательных навыков"],
  "nice_to_have": ["список желательных навыков"],
  "experience_required": "например '3-5 лет' или null",
  "salary_to": "верхняя граница зарплаты или null",
  "work_format": "офис / удалёнка / гибрид"
}
```

### Требования
- Используйте `response_format={'type': 'json_object'}`
- Системный промпт должен содержать слово `JSON` и описание схемы
- Оберните вызов в `try/except`
- Проверьте функцию на всех пяти тестовых вакансиях ниже

### Бонус ⭐
Напишите функцию `match_candidates(vacancy: dict, candidates: list) -> list`, которая по уже разобранным данным подбирает подходящих кандидатов из списка `results` (совпадение хотя бы 2 навыков из `required_skills`).

In [26]:
# Тестовые вакансии
vacancy_1 = """
Компания TechFlow ищет Senior Python Developer.
Обязательно: Python 5+ лет, FastAPI или Django, PostgreSQL, Docker.
Желательно: Kubernetes, опыт с микросервисами, Redis.
Условия: гибрид (Москва), зарплата до 400 000 рублей, ДМС.
"""

vacancy_2 = """
Ищем Middle Data Scientist в e-commerce стартап.
Требования: Python, pandas, scikit-learn, опыт построения ML-моделей от 2 лет.
Плюсом будет: PyTorch, опыт с рекомендательными системами, A/B-тесты.
Полная удалёнка, зарплата до 220 000. Команда 5 человек.
"""

vacancy_3 = """
Frontend Developer (React) — продуктовая компания в сфере EdTech.
Must have: React, TypeScript, REST API, опыт от 1 года.
Nice to have: Next.js, тестирование (Jest), Storybook.
Офис в СПб или удалёнка, вилка 150 000 – 200 000.
"""


def parse_vacancy(text: str, attempts: int = 3) -> dict | None:
    """
    Извлекает структурированные данные из текста вакансии.

    Args:
        text: Текст вакансии в произвольном формате

    Returns:
        Словарь с полями position, company, required_skills,
        nice_to_have, experience_required, salary_to, work_format
        или None при ошибке
    """
    # Схема вакансии: 7 полей, отсутствующие данные — null
    vacancy_schema = {
        'type': 'object',
        'properties': {
            'position': {'type': 'string'},
            'company': {'type': ['string', 'null']},
            'required_skills': {'type': 'array', 'items': {'type': 'string'}},
            'nice_to_have': {'type': 'array', 'items': {'type': 'string'}},
            'experience_required': {'type': ['string', 'null']},
            'salary_to': {'type': ['number', 'null']},
            'work_format': {'type': 'string'},
        },
        'required': [
            'position', 'company', 'required_skills', 'nice_to_have',
            'experience_required', 'salary_to', 'work_format',
        ],
        'additionalProperties': False,
    }

    system_prompt = """
Ты — HR-ассистент. Извлекай данные из вакансии и возвращай строго в формате JSON.
required_skills и nice_to_have — списки названий технологий: по одному названию
в каждом элементе, без кавычек и без пояснений. Альтернативы разделяй на отдельные
элементы: 'FastAPI или Django' — это два элемента 'FastAPI' и 'Django'.
Уточнение опыта ('5+ лет') — не часть названия навыка, его место в experience_required.
experience_required — строка вида '3-5 лет' или 'от 2 лет'.
salary_to — верхняя граница зарплаты (число) или null, если не указана.
Если поле в тексте отсутствует — верни null.
"""

    chat = Chat(
        model='GigaChat-3-Lightning',  # выбрана по итогам бенчмарка раздела 6
        messages=[
            Messages(role=MessagesRole.SYSTEM, content=system_prompt),
            Messages(role=MessagesRole.USER, content=f'Обработай вакансию:\n{text}'),
        ],
        response_format={
            'type': 'json_schema',
            'schema': vacancy_schema,
            'strict': True,
        },
    )

    for attempt in range(1, attempts + 1):
        try:
            response = client.chat(chat)
            return json.loads(response.choices[0].message.content)
        except RateLimitError as e:
            wait = getattr(e, 'retry_after', None) or 5
            print(f'Лимит запросов (попытка {attempt}/{attempts}). Ждём {wait} сек...')
            time.sleep(wait)
        except Exception as e:
            print(f'Ошибка: {e}')
            return None
    return None


# Тестирование
for i, vacancy in enumerate([vacancy_1, vacancy_2, vacancy_3], 1):
    print('=' * 55)
    print(f'Вакансия {i}')
    print('=' * 55)
    result = parse_vacancy(vacancy)
    if result:
        print(json.dumps(result, ensure_ascii=False, indent=2))
        print()

Вакансия 1
{
  "position": "Senior Python Developer",
  "company": "TechFlow",
  "required_skills": [
    "Python",
    "FastAPI",
    "Django",
    "PostgreSQL",
    "Docker"
  ],
  "nice_to_have": [
    "Kubernetes",
    "микросервисы",
    "Redis"
  ],
  "experience_required": "5+ лет",
  "salary_to": 400000,
  "work_format": "гибрид (Москва)"
}

Вакансия 2
{
  "position": "Middle Data Scientist",
  "company": "e-commerce стартап",
  "required_skills": [
    "Python",
    "pandas",
    "scikit-learn"
  ],
  "nice_to_have": [
    "PyTorch",
    "рекомендательные системы",
    "A/B-тесты"
  ],
  "experience_required": "от 2 лет",
  "salary_to": 220000,
  "work_format": "полная удалёнка"
}

Вакансия 3
{
  "position": "Frontend Developer (React)",
  "company": "продуктовая компания в сфере EdTech",
  "required_skills": [
    "React",
    "TypeScript",
    "REST API"
  ],
  "nice_to_have": [
    "Next.js",
    "Jest",
    "Storybook"
  ],
  "experience_required": "от 1 года",
  "salary_t

In [27]:
def match_candidates(vacancy: dict, candidates: list) -> list:
    """
    Подбирает кандидатов из списка по совпадению навыков.

    Args:
        vacancy: Разобранная вакансия (результат parse_vacancy)
        candidates: Список разобранных резюме (результаты parse_resume)

    Returns:
        Список подходящих кандидатов (>= 2 совпадений) с полем matched_skills
    """
    required = [s.lower() for s in vacancy.get('required_skills', [])]
    matched = []

    for cand in candidates:
        cand_skills = [s.lower() for s in cand.get('skills', [])]
        # Подстрочный матчинг — из-за составных навыков вида "Python (Django, FastAPI)"
        hits = [
            req for req in required
            if any(req in skill for skill in cand_skills)
        ]
        if len(hits) >= 2:
            entry = dict(cand)  # копия, чтобы не портить исходные results
            entry['matched_skills'] = hits
            matched.append(entry)

    return matched


# Демонстрация: матчинг по всем трём вакансиям
vacancies = [vacancy_1, vacancy_2, vacancy_3]
for i, vac_text in enumerate(vacancies, 1):
    vacancy = parse_vacancy(vac_text)
    if not vacancy:
        continue
    print(f"Вакансия {i}: {vacancy['position']}")
    for m in match_candidates(vacancy, results):
        print(f"   → {m['name']}: совпало {m['matched_skills']}")
    print()

Вакансия 1: Senior Python Developer
   → Алексей Новиков: совпало ['python', 'fastapi', 'django', 'postgresql', 'docker']

Вакансия 2: Middle Data Scientist
   → Мария Иванова: совпало ['python', 'pandas', 'scikit-learn']

Вакансия 3: Frontend Developer (React)
   → Дмитрий Козлов: совпало ['react', 'typescript']



---
## Чек-лист перед сдачей

- [ ] Все ячейки выполняются без ошибок (Kernel → Restart & Run All)
- [ ] Раздел 5: выполнены все 5 заданий по фильтрации
- [ ] Раздел 6: таблица сравнения моделей заполнена, письменные ответы добавлены
- [ ] Функция `parse_vacancy()` реализована и возвращает корректный JSON для всех 5 вакансий
- [ ] Все 7 полей схемы присутствуют в ответе (или `null`, если данных нет)
- [ ] Используется `response_format={'type': 'json_object'}`
- [ ] В коде есть обработка ошибок `try/except`
- [ ] API-ключ не хранится в коде — только через `.env`
- [ ] (Бонус) Функция `match_candidates()` реализована
